In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/integrated_data.csv', parse_dates=['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

trend_cols = ['BTC_google_trend', 'ETH_google_trend', 'BNB_google_trend', 
              'ADA_google_trend', 'SOL_google_trend', 'DOGE_google_trend']

symbol_to_trend = {
    'BTC': 'BTC_google_trend', 'ETH': 'ETH_google_trend', 'BNB': 'BNB_google_trend',
    'ADA': 'ADA_google_trend', 'SOL': 'SOL_google_trend', 'DOGE': 'DOGE_google_trend'
}

def compute_features(group, symbol):
    features = {}
    
    # Price features
    features['ret_1d'] = group['close'].pct_change()
    features['ret_7d'] = group['close'].pct_change(7)
    features['ret_14d'] = group['close'].pct_change(14)
    features['ret_30d'] = group['close'].pct_change(30)
    features['log_ret'] = np.log(group['close'] / group['close'].shift(1).replace(0, np.nan))
    features['ma_7'] = group['close'].rolling(7).mean()
    features['ma_14'] = group['close'].rolling(14).mean()
    features['ma_30'] = group['close'].rolling(30).mean()
    features['ma_60'] = group['close'].rolling(60).mean()
    features['ma_ratio_7_30'] = features['ma_7'] / features['ma_30']
    features['ma_ratio_30_60'] = features['ma_30'] / features['ma_60']
    features['std_7'] = group['close'].rolling(7).std()
    features['std_14'] = group['close'].rolling(14).std()
    features['std_30'] = group['close'].rolling(30).std()
    features['zscore_30'] = (group['close'] - features['ma_30']) / features['std_30']
    features['zscore_60'] = (group['close'] - features['ma_60']) / group['close'].rolling(60).std()
    
    # Volatility
    features['daily_range'] = (group['high'] - group['low']) / group['close']
    features['daily_range_pct'] = (group['high'] - group['low']) / group['open']
    features['atr_7'] = features['daily_range'].rolling(7).mean()
    features['atr_14'] = features['daily_range'].rolling(14).mean()
    features['atr_30'] = features['daily_range'].rolling(30).mean()
    features['volatility_trend'] = features['atr_7'] / features['atr_30']
    features['realized_vol_7'] = features['log_ret'].rolling(7).std() * np.sqrt(365)
    features['realized_vol_30'] = features['log_ret'].rolling(30).std() * np.sqrt(365)
    
    # Volume
    features['volume_ma_7'] = group['volume'].rolling(7).mean()
    features['volume_ma_14'] = group['volume'].rolling(14).mean()
    features['volume_ma_30'] = group['volume'].rolling(30).mean()
    features['volume_ratio_7'] = group['volume'] / features['volume_ma_7']
    features['volume_ratio_30'] = group['volume'] / features['volume_ma_30']
    features['volume_change_7d'] = group['volume'].pct_change(7)
    features['volume_std_30'] = group['volume'].rolling(30).std()
    features['volume_zscore'] = (group['volume'] - features['volume_ma_30']) / features['volume_std_30']
    features['price_volume_corr_30'] = group['close'].rolling(30).corr(group['volume'])
    
    # RSI
    delta = group['close'].diff()
    gain = delta.clip(lower=0)
    loss = delta.clip(upper=0).abs()
    features['rsi_7'] = 100 - 100 / (1 + gain.rolling(7).mean() / loss.rolling(7).mean().replace(0, np.nan))
    features['rsi_14'] = 100 - 100 / (1 + gain.rolling(14).mean() / loss.rolling(14).mean().replace(0, np.nan))
    features['rsi_30'] = 100 - 100 / (1 + gain.rolling(30).mean() / loss.rolling(30).mean().replace(0, np.nan))
    
    # Momentum
    features['momentum_7'] = group['close'] - group['close'].shift(7)
    features['momentum_14'] = group['close'] - group['close'].shift(14)
    features['momentum_30'] = group['close'] - group['close'].shift(30)
    features['roc_7'] = (group['close'] - group['close'].shift(7)) / group['close'].shift(7) * 100
    features['roc_14'] = (group['close'] - group['close'].shift(14)) / group['close'].shift(14) * 100
    features['roc_30'] = (group['close'] - group['close'].shift(30)) / group['close'].shift(30) * 100
    features['price_position_30'] = (group['close'] - group['close'].rolling(30).min()) / (group['close'].rolling(30).max() - group['close'].rolling(30).min())
    features['price_position_60'] = (group['close'] - group['close'].rolling(60).min()) / (group['close'].rolling(60).max() - group['close'].rolling(60).min())
    
    # Google Trends
    if symbol in symbol_to_trend:
        trend = group[symbol_to_trend[symbol]]
        features['own_trend'] = trend
        features['own_trend_ma_7'] = trend.rolling(7).mean()
        features['own_trend_ma_14'] = trend.rolling(14).mean()
        features['own_trend_ma_30'] = trend.rolling(30).mean()
        features['own_trend_std_30'] = trend.rolling(30).std()
        features['own_trend_zscore'] = (trend - features['own_trend_ma_30']) / features['own_trend_std_30']
        features['own_trend_change_1d'] = trend.diff()
        features['own_trend_change_7d'] = trend.diff(7)
        features['own_trend_momentum'] = features['own_trend_ma_7'] - features['own_trend_ma_30']
        features['trend_price_corr_30'] = trend.rolling(30).corr(group['close'])
        features['trend_volume_corr_30'] = trend.rolling(30).corr(group['volume'])
        features['trend_price_divergence'] = features['own_trend_zscore'] - features['zscore_30']
    
    for col in trend_cols:
        short_name = col.replace('_google_trend', '')
        features[f'{short_name}_trend'] = group[col]
        features[f'{short_name}_trend_ma_7'] = group[col].rolling(7).mean()
        features[f'{short_name}_trend_change_7d'] = group[col].diff(7)
    
    features['btc_trend_ma_30'] = group['BTC_google_trend'].rolling(30).mean()
    features['btc_trend_zscore'] = (group['BTC_google_trend'] - features['btc_trend_ma_30']) / group['BTC_google_trend'].rolling(30).std()
    
    # Sentiment
    features['fear_greed'] = group['value']
    features['fear_greed_ma_7'] = group['value'].rolling(7).mean()
    features['fear_greed_ma_14'] = group['value'].rolling(14).mean()
    features['fear_greed_ma_30'] = group['value'].rolling(30).mean()
    features['fear_greed_std_30'] = group['value'].rolling(30).std()
    features['fear_greed_zscore'] = (group['value'] - features['fear_greed_ma_30']) / features['fear_greed_std_30']
    features['fear_greed_change_1d'] = group['value'].diff()
    features['fear_greed_change_7d'] = group['value'].diff(7)
    features['fear_greed_change_14d'] = group['value'].diff(14)
    features['fear_greed_momentum'] = features['fear_greed_ma_7'] - features['fear_greed_ma_30']
    features['extreme_greed'] = (group['value'] > 75).astype(int)
    features['extreme_fear'] = (group['value'] < 25).astype(int)
    features['greed_streak'] = (group['value'] > 60).astype(int).groupby((group['value'] <= 60).cumsum()).cumsum()
    features['fear_streak'] = (group['value'] < 40).astype(int).groupby((group['value'] >= 40).cumsum()).cumsum()
    features['sentiment_price_corr_30'] = group['value'].rolling(30).corr(features['ret_1d'])
    features['sentiment_price_divergence'] = features['fear_greed_ma_7'] - features['zscore_30'] * 25 + 50
    
    # Macro
    features['fed_funds_change_1d'] = group['fed_funds'].diff()
    features['fed_funds_change_7d'] = group['fed_funds'].diff(7)
    features['fed_funds_change_30d'] = group['fed_funds'].diff(30)
    features['fed_funds_ma_30'] = group['fed_funds'].rolling(30).mean()
    features['treasury_10y_change_1d'] = group['treasury_10y'].diff()
    features['treasury_10y_change_7d'] = group['treasury_10y'].diff(7)
    features['treasury_10y_change_30d'] = group['treasury_10y'].diff(30)
    features['treasury_10y_ma_30'] = group['treasury_10y'].rolling(30).mean()
    features['yield_spread'] = group['treasury_10y'] - group['fed_funds']
    features['yield_spread_change'] = features['yield_spread'].diff(7)
    features['vix_ma_7'] = group['vix'].rolling(7).mean()
    features['vix_ma_30'] = group['vix'].rolling(30).mean()
    features['vix_std_30'] = group['vix'].rolling(30).std()
    features['vix_zscore'] = (group['vix'] - features['vix_ma_30']) / features['vix_std_30']
    features['vix_change_1d'] = group['vix'].diff()
    features['vix_change_7d'] = group['vix'].diff(7)
    features['vix_regime'] = (group['vix'] > 20).astype(int) + (group['vix'] > 30).astype(int)
    features['m2_growth_30d'] = group['m2'].pct_change(30)
    features['m2_growth_60d'] = group['m2'].pct_change(60)
    features['m2_growth_90d'] = group['m2'].pct_change(90)
    features['m2_ma_30'] = group['m2'].rolling(30).mean()
    features['m2_acceleration'] = features['m2_growth_30d'] - features['m2_growth_30d'].shift(30)
    features['cpi_change_30d'] = group['cpi'].diff(30)
    features['cpi_ma_30'] = group['cpi'].rolling(30).mean()
    features['real_rate'] = group['treasury_10y'] - group['cpi']
    features['real_rate_change'] = features['real_rate'].diff(30)
    features['price_vix_corr_30'] = group['close'].rolling(30).corr(group['vix'])
    features['price_m2_corr_60'] = group['close'].rolling(60).corr(group['m2'])
    features['risk_off_signal'] = ((group['vix'] > features['vix_ma_30']) & (group['treasury_10y'] < features['treasury_10y_ma_30'])).astype(int)
    
    # Lagged features
    for lag in [1, 3, 7, 14]:
        for col in ['ret_1d', 'ret_7d', 'volume_ratio_30', 'fear_greed', 'rsi_14']:
            features[f'{col}_lag_{lag}'] = features[col].shift(lag)
    
    # Time features
    features['day_of_week'] = group['date'].dt.dayofweek
    features['month'] = group['date'].dt.month
    features['quarter'] = group['date'].dt.quarter
    features['year'] = group['date'].dt.year
    features['is_month_start'] = group['date'].dt.is_month_start.astype(int)
    features['is_month_end'] = group['date'].dt.is_month_end.astype(int)
    
    return pd.DataFrame(features, index=group.index)

print("Computing features per coin...")
result_list = []
base_cols = ['date', 'symbol', 'open', 'high', 'low', 'close', 'volume'] + trend_cols + ['fed_funds', 'treasury_10y', 'vix', 'm2', 'cpi', 'value', 'value_classification']

for symbol, group in df.groupby('symbol'):
    feat = compute_features(group, symbol)
    combined = pd.concat([group[base_cols].reset_index(drop=True), feat.reset_index(drop=True)], axis=1)
    combined = combined.iloc[90:]  # Drop warmup period
    
    # NA handling with median imputation per coin
    num_cols = combined.select_dtypes(include=[np.number]).columns
    combined[num_cols] = combined[num_cols].fillna(combined[num_cols].median())
    
    result_list.append(combined)

feature_df = pd.concat(result_list, ignore_index=True)

feature_df.to_csv('data/df_features.csv', index=False)

print(f"Shape: {feature_df.shape}")
print(f"Coins: {feature_df['symbol'].unique()}")
print(f"Date range: {feature_df['date'].min()} to {feature_df['date'].max()}")
print(f"Rows per coin: {feature_df.groupby('symbol').size().to_dict()}")
print(f"NaNs remaining: {feature_df.select_dtypes(include=[np.number]).isnull().sum().sum()}")

Computing features per coin...
Shape: (12825, 167)
Coins: <StringArray>
['ADA', 'BNB', 'BTC', 'DOGE', 'ETH', 'SOL']
Length: 6, dtype: str
Date range: 2018-07-30 00:00:00 to 2025-01-01 00:00:00
Rows per coin: {'ADA': 2348, 'BNB': 2348, 'BTC': 2348, 'DOGE': 1918, 'ETH': 2348, 'SOL': 1515}
NaNs remaining: 0
